# Operational Compliance Dashboard: Little Oaks Academy Staffing Ratios

This notebook creates a realistic operational simulation dataset for a preschool staffing-ratio dashboard. The dataset is designed to mimic daily classroom check-in patterns, active staff coverage, and compliance status across multiple age groups and time blocks.

In [1]:
import pandas as pd
import numpy as np
import math

## Define the classroom structure

We create a realistic academy setup with 8 classrooms across 4 age groups:
- Infants
- Toddlers
- Preschool
- Pre-K

In [2]:
classrooms = [
    {"Classroom_ID": "INF-1", "Classroom_Name": "Infant Room A", "Age_Group": "Infant", "Licensed_Capacity": 8,  "Ratio_Limit": 4},
    {"Classroom_ID": "INF-2", "Classroom_Name": "Infant Room B", "Age_Group": "Infant", "Licensed_Capacity": 8,  "Ratio_Limit": 4},
    {"Classroom_ID": "TOD-1", "Classroom_Name": "Toddler Room A", "Age_Group": "Toddler", "Licensed_Capacity": 12, "Ratio_Limit": 6},
    {"Classroom_ID": "TOD-2", "Classroom_Name": "Toddler Room B", "Age_Group": "Toddler", "Licensed_Capacity": 12, "Ratio_Limit": 6},
    {"Classroom_ID": "PRE-1", "Classroom_Name": "Preschool Room A", "Age_Group": "Preschool", "Licensed_Capacity": 20, "Ratio_Limit": 10},
    {"Classroom_ID": "PRE-2", "Classroom_Name": "Preschool Room B", "Age_Group": "Preschool", "Licensed_Capacity": 20, "Ratio_Limit": 10},
    {"Classroom_ID": "PK-1",  "Classroom_Name": "Pre-K Room A", "Age_Group": "Pre-K", "Licensed_Capacity": 24, "Ratio_Limit": 12},
    {"Classroom_ID": "PK-2",  "Classroom_Name": "Pre-K Room B", "Age_Group": "Pre-K", "Licensed_Capacity": 24, "Ratio_Limit": 12},
]

classrooms_df = pd.DataFrame(classrooms)
classrooms_df

,Classroom_ID,Classroom_Name,Age_Group,Licensed_Capacity,Ratio_Limit
0,INF-1,Infant Room A,Infant,8,4
1,INF-2,Infant Room B,Infant,8,4
2,TOD-1,Toddler Room A,Toddler,12,6
3,TOD-2,Toddler Room B,Toddler,12,6
4,PRE-1,Preschool Room A,Preschool,20,10
5,PRE-2,Preschool Room B,Preschool,20,10
6,PK-1,Pre-K Room A,Pre-K,24,12
7,PK-2,Pre-K Room B,Pre-K,24,12


## Define the school days and time blocks

We simulate one operational week across realistic check-in periods:
- morning drop-off
- midday steady state
- afternoon pickup

In [3]:
dates = pd.date_range(start="2025-04-07", periods=5, freq="D")  # Monday to Friday

time_blocks = [
    "07:30",
    "08:00",
    "08:30",
    "09:00",
    "10:30",
    "12:00",
    "14:00",
    "16:00",
    "17:00"
]

dates, time_blocks

(DatetimeIndex(['2025-04-07', '2025-04-08', '2025-04-09', '2025-04-10',
                '2025-04-11'],
               dtype='datetime64[ns]', freq='D'),
 ['07:30',
  '08:00',
  '08:30',
  '09:00',
  '10:30',
  '12:00',
  '14:00',
  '16:00',
  '17:00'])

## Create baseline enrollment behavior by age group

This gives each age group a realistic attendance pattern throughout the day.

In [4]:
attendance_profiles = {
    "Infant":   [0.55, 0.75, 0.90, 1.00, 1.00, 0.95, 0.90, 0.75, 0.55],
    "Toddler":  [0.50, 0.70, 0.90, 1.00, 1.00, 0.95, 0.90, 0.70, 0.45],
    "Preschool":[0.45, 0.65, 0.85, 1.00, 1.00, 0.98, 0.92, 0.65, 0.35],
    "Pre-K":    [0.40, 0.60, 0.85, 1.00, 1.00, 0.98, 0.90, 0.60, 0.30],
}

attendance_profiles

{'Infant': [0.55, 0.75, 0.9, 1.0, 1.0, 0.95, 0.9, 0.75, 0.55],
 'Toddler': [0.5, 0.7, 0.9, 1.0, 1.0, 0.95, 0.9, 0.7, 0.45],
 'Preschool': [0.45, 0.65, 0.85, 1.0, 1.0, 0.98, 0.92, 0.65, 0.35],
 'Pre-K': [0.4, 0.6, 0.85, 1.0, 1.0, 0.98, 0.9, 0.6, 0.3]}

## Build the daily operational rows

For each classroom, day, and time block, we simulate:
- student count
- active staff count
- required staff
- current ratio
- pass/fail compliance

In [5]:
rows = []

rng = np.random.default_rng(42)

for date in dates:
    weekday = date.day_name()

    for room in classrooms:
        capacity = room["Licensed_Capacity"]
        ratio_limit = room["Ratio_Limit"]
        age_group = room["Age_Group"]
        profile = attendance_profiles[age_group]

        # classroom-specific baseline occupancy
        base_enrollment = int(round(capacity * rng.uniform(0.75, 0.98)))

        for idx, time_str in enumerate(time_blocks):
            attendance_factor = profile[idx]

            # simulate current students
            student_count = int(round(base_enrollment * attendance_factor))

            # small random fluctuation
            student_count += int(rng.integers(-1, 2))
            student_count = max(0, min(student_count, capacity))

            # required staff from regulation
            required_staff = math.ceil(student_count / ratio_limit) if student_count > 0 else 0

            # simulate actual staff with realistic dips around transitions
            staff_variation = 0

            if time_str in ["08:00", "08:30", "16:00", "17:00"]:
                staff_variation = int(rng.choice([-1, 0, 0, 1]))
            elif time_str in ["12:00", "14:00"]:
                staff_variation = int(rng.choice([-1, 0, 0]))
            else:
                staff_variation = int(rng.choice([0, 0, 1]))

            active_staff = max(1, required_staff + staff_variation) if student_count > 0 else 0

            current_ratio = round(student_count / active_staff, 2) if active_staff > 0 else 0
            compliance_status = "Pass" if active_staff >= required_staff else "Fail"
            alert_flag = "Clear" if compliance_status == "Pass" else "Alert"

            rows.append({
                "Date": date.date(),
                "Weekday": weekday,
                "Time_of_Day": time_str,
                "Classroom_ID": room["Classroom_ID"],
                "Classroom_Name": room["Classroom_Name"],
                "Age_Group": age_group,
                "Licensed_Capacity": capacity,
                "Student_Count": student_count,
                "Active_Staff_Count": active_staff,
                "Ratio_Limit": ratio_limit,
                "Required_Staff": required_staff,
                "Current_Ratio": current_ratio,
                "Compliance_Status": compliance_status,
                "Alert_Flag": alert_flag
            })

sim_df = pd.DataFrame(rows)
sim_df.head(20)

,Date,Weekday,Time_of_Day,Classroom_ID,Classroom_Name,Age_Group,Licensed_Capacity,Student_Count,Active_Staff_Count,Ratio_Limit,Required_Staff,Current_Ratio,Compliance_Status,Alert_Flag
0,2025-04-07,Monday,07:30,INF-1,Infant Room A,Infant,8,4,1,4,1,4.00,Pass,Clear
1,2025-04-07,Monday,08:00,INF-1,Infant Room A,Infant,8,5,3,4,2,1.67,Pass,Clear
2,2025-04-07,Monday,08:30,INF-1,Infant Room A,Infant,8,5,2,4,2,2.50,Pass,Clear
3,2025-04-07,Monday,09:00,INF-1,Infant Room A,Infant,8,6,2,4,2,3.00,Pass,Clear
4,2025-04-07,Monday,10:30,INF-1,Infant Room A,Infant,8,7,3,4,2,2.33,Pass,Clear
5,2025-04-07,Monday,12:00,INF-1,Infant Room A,Infant,8,8,2,4,2,4.00,Pass,Clear
6,2025-04-07,Monday,14:00,INF-1,Infant Room A,Infant,8,7,2,4,2,3.50,Pass,Clear
7,2025-04-07,Monday,16:00,INF-1,Infant Room A,Infant,8,5,1,4,2,5.00,Fail,Alert
8,2025-04-07,Monday,17:00,INF-1,Infant Room A,Infant,8,5,2,4,2,2.50,Pass,Clear
9,2025-04-07,Monday,07:30,INF-2,Infant Room B,Infant,8,3,2,4,1,1.50,Pass,Clear


## Check the dataset shape

This confirms we created a realistic operational-size dataset.

In [6]:
print(sim_df.shape)

(360, 14)


## Check compliance counts

This helps confirm that the simulation includes both compliant and non-compliant states.

In [7]:
sim_df["Compliance_Status"].value_counts()

,count
Compliance_Status,
Pass,305
Fail,55


## Preview classroom ratio behavior

In [8]:
sim_df[[
    "Date",
    "Time_of_Day",
    "Classroom_Name",
    "Age_Group",
    "Student_Count",
    "Active_Staff_Count",
    "Ratio_Limit",
    "Required_Staff",
    "Current_Ratio",
    "Compliance_Status"
]].head(20)

,Date,Time_of_Day,Classroom_Name,Age_Group,Student_Count,Active_Staff_Count,Ratio_Limit,Required_Staff,Current_Ratio,Compliance_Status
0,2025-04-07,07:30,Infant Room A,Infant,4,1,4,1,4.00,Pass
1,2025-04-07,08:00,Infant Room A,Infant,5,3,4,2,1.67,Pass
2,2025-04-07,08:30,Infant Room A,Infant,5,2,4,2,2.50,Pass
3,2025-04-07,09:00,Infant Room A,Infant,6,2,4,2,3.00,Pass
4,2025-04-07,10:30,Infant Room A,Infant,7,3,4,2,2.33,Pass
5,2025-04-07,12:00,Infant Room A,Infant,8,2,4,2,4.00,Pass
6,2025-04-07,14:00,Infant Room A,Infant,7,2,4,2,3.50,Pass
7,2025-04-07,16:00,Infant Room A,Infant,5,1,4,2,5.00,Fail
8,2025-04-07,17:00,Infant Room A,Infant,5,2,4,2,2.50,Pass
9,2025-04-07,07:30,Infant Room B,Infant,3,2,4,1,1.50,Pass


# Step 2: Create executive and drill-down helper fields

In this step, we derive a few dashboard-ready fields that support executive status, time-of-day analysis, and ratio monitoring in Tableau.

## Create a time block group

This helps the dashboard summarize peak drop-off, midday, and pick-up periods more clearly.

In [9]:
def map_time_group(t):
    if t in ["07:30", "08:00", "08:30", "09:00"]:
        return "Drop-off Window"
    elif t in ["10:30", "12:00", "14:00"]:
        return "Midday"
    else:
        return "Pick-up Window"

sim_df["Time_Group"] = sim_df["Time_of_Day"].apply(map_time_group)
sim_df[["Time_of_Day", "Time_Group"]].drop_duplicates().sort_values("Time_of_Day")

,Time_of_Day,Time_Group
0,07:30,Drop-off Window
1,08:00,Drop-off Window
2,08:30,Drop-off Window
3,09:00,Drop-off Window
4,10:30,Midday
5,12:00,Midday
6,14:00,Midday
7,16:00,Pick-up Window
8,17:00,Pick-up Window


## Create classroom utilization

This shows how full each room is relative to licensed capacity.

In [10]:
sim_df["Capacity_Utilization_Pct"] = (
    sim_df["Student_Count"] / sim_df["Licensed_Capacity"] * 100
).round(1)

sim_df[["Student_Count", "Licensed_Capacity", "Capacity_Utilization_Pct"]].head()

,Student_Count,Licensed_Capacity,Capacity_Utilization_Pct
0,4,8,50.0
1,5,8,62.5
2,5,8,62.5
3,6,8,75.0
4,7,8,87.5


## Create staffing coverage gap

Negative values indicate understaffing relative to required staffing.

In [11]:
sim_df["Staffing_Gap"] = sim_df["Active_Staff_Count"] - sim_df["Required_Staff"]
sim_df[["Active_Staff_Count", "Required_Staff", "Staffing_Gap"]].head()

,Active_Staff_Count,Required_Staff,Staffing_Gap
0,1,1,0
1,3,2,1
2,2,2,0
3,2,2,0
4,3,2,1


## Create a clearer alert label

This gives a user-friendly operational message for the dashboard.

In [12]:
def alert_label(row):
    if row["Compliance_Status"] == "Pass":
        return "Compliant"
    else:
        return "Non-Compliant"

sim_df["Alert_Label"] = sim_df.apply(alert_label, axis=1)
sim_df[["Compliance_Status", "Alert_Label"]].drop_duplicates()

,Compliance_Status,Alert_Label
0,Pass,Compliant
7,Fail,Non-Compliant


## Create a facility-wide executive flag by timestamp

If any classroom fails at a given time, the facility is marked Alert for that timestamp.

In [13]:
facility_status = (
    sim_df.groupby(["Date", "Weekday", "Time_of_Day", "Time_Group"], as_index=False)
    .agg({"Compliance_Status": lambda x: "Fail" if "Fail" in list(x) else "Pass"})
    .rename(columns={"Compliance_Status": "Facility_Status"})
)

sim_df = sim_df.merge(
    facility_status,
    on=["Date", "Weekday", "Time_of_Day", "Time_Group"],
    how="left"
)

sim_df[[
    "Date", "Time_of_Day", "Classroom_Name", "Compliance_Status", "Facility_Status"
]].head(20)

,Date,Time_of_Day,Classroom_Name,Compliance_Status,Facility_Status
0,2025-04-07,07:30,Infant Room A,Pass,Pass
1,2025-04-07,08:00,Infant Room A,Pass,Pass
2,2025-04-07,08:30,Infant Room A,Pass,Fail
3,2025-04-07,09:00,Infant Room A,Pass,Pass
4,2025-04-07,10:30,Infant Room A,Pass,Pass
5,2025-04-07,12:00,Infant Room A,Pass,Fail
6,2025-04-07,14:00,Infant Room A,Pass,Fail
7,2025-04-07,16:00,Infant Room A,Fail,Fail
8,2025-04-07,17:00,Infant Room A,Pass,Pass
9,2025-04-07,07:30,Infant Room B,Pass,Pass


## Preview the enhanced dataset

In [14]:
print(sim_df.shape)
sim_df.head()

(360, 19)


,Date,Weekday,Time_of_Day,Classroom_ID,Classroom_Name,Age_Group,Licensed_Capacity,Student_Count,Active_Staff_Count,Ratio_Limit,Required_Staff,Current_Ratio,Compliance_Status,Alert_Flag,Time_Group,Capacity_Utilization_Pct,Staffing_Gap,Alert_Label,Facility_Status
0,2025-04-07,Monday,07:30,INF-1,Infant Room A,Infant,8,4,1,4,1,4.00,Pass,Clear,Drop-off Window,50.0,0,Compliant,Pass
1,2025-04-07,Monday,08:00,INF-1,Infant Room A,Infant,8,5,3,4,2,1.67,Pass,Clear,Drop-off Window,62.5,1,Compliant,Pass
2,2025-04-07,Monday,08:30,INF-1,Infant Room A,Infant,8,5,2,4,2,2.50,Pass,Clear,Drop-off Window,62.5,0,Compliant,Fail
3,2025-04-07,Monday,09:00,INF-1,Infant Room A,Infant,8,6,2,4,2,3.00,Pass,Clear,Drop-off Window,75.0,0,Compliant,Pass
4,2025-04-07,Monday,10:30,INF-1,Infant Room A,Infant,8,7,3,4,2,2.33,Pass,Clear,Midday,87.5,1,Compliant,Pass


## Save the Tableau-ready dataset

In [15]:
sim_df.to_csv("little_oaks_staffing_ready.csv", index=False)
print("Saved as little_oaks_staffing_ready.csv")

Saved as little_oaks_staffing_ready.csv


In [16]:
from google.colab import files
files.download("little_oaks_staffing_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>